# Oracle Retrieval-Mode Evaluation Metrics

This notebook compares the three Oracle retrieval modes using compact metrics instead of a large parameter grid.

Metrics shown:

- latency in milliseconds
- local result count
- foreign Tavily result count
- rows inserted per run
- total rows for that query
- short interpretation

The final cell deletes the benchmark rows so repeated runs stay consistent.

## Setup

Run this cell first. It loads `.env`, connects to Oracle, prepares the demo table, and imports shared notebook helpers.

In [1]:
from pathlib import Path
import sys

helper_path = None
for root in [Path.cwd(), *Path.cwd().parents]:
    candidate = root / "examples" / "oracle" / "demo_helpers.py"
    if candidate.exists():
        helper_path = candidate
        break

if helper_path is None:
    candidate = Path.cwd() / "demo_helpers.py"
    if candidate.exists():
        helper_path = candidate

if helper_path is None:
    raise RuntimeError("Could not find examples/oracle/demo_helpers.py. Start Jupyter from the repository root or examples/oracle.")

sys.path.insert(0, str(helper_path.parent))
from demo_helpers import *

print("Using helper:", helper_path)

Loaded environment from /Users/anishraj/Desktop/Projects/Pod4_demo/tavily-python/.env
Keeping proxy variables because TAVILY_USE_ENV_PROXY=1
Connected to Oracle. Table: TAVILY_DOCUMENTS
Table exists. Schema is ready.
Demo helpers are ready. Scores are ranking signals, not probabilities.
Using helper: /Users/anishraj/Desktop/Projects/Pod4_demo/tavily-python/examples/oracle/demo_helpers.py


## Edit benchmark queries here

Change the `query` strings inside `scenarios` if you want to benchmark a different topic. Keep each scenario query different so row counts stay easy to read.

In [2]:
def run_and_measure(mode, client, query, run_label, sleep_before=0):
    if sleep_before:
        time.sleep(sleep_before)
    rows_before = count_rows_for_query(query)
    started = time.perf_counter()
    results = client.search(
        query,
        max_results=3,
        max_local=3,
        max_foreign=3,
        save_foreign=True,
        **TAVILY_SEARCH_OPTIONS,
    )
    latency_ms = (time.perf_counter() - started) * 1000
    rows_after = count_rows_for_query(query)
    origins = Counter(result.get("origin", "unknown") for result in results)
    return {
        "mode": mode,
        "run": run_label,
        "latency_ms": f"{latency_ms:.1f}",
        "local": origins.get("local", 0),
        "foreign": origins.get("foreign", 0),
        "rows_delta": rows_after - rows_before,
        "rows_total": rows_after,
        "interpretation": "Oracle reused cache/memory" if origins.get("local", 0) else "Tavily supplied fresh results",
    }

scenarios = [
    {
        "mode": "hybrid_search",
        "query": "Oracle Database vector search evaluation hybrid mode",
        "client": make_client("hybrid_search", persistence_depth="cache_plus_memory"),
        "sleep_second_run": 0,
    },
    {
        "mode": "freshness_cache",
        "query": "Oracle Database vector search evaluation freshness cache",
        "client": make_client("freshness_cache", persistence_depth="cache_only", cache_score_threshold=-1.0),
        "sleep_second_run": 0,
    },
    {
        "mode": "cache_then_memory",
        "query": "Oracle Database vector search evaluation cache then memory",
        "client": make_client(
            "cache_then_memory",
            persistence_depth="cache_plus_memory",
            cache_ttl_seconds=1,
            cache_score_threshold=-1.0,
            memory_score_threshold=-1.0,
        ),
        "sleep_second_run": 2,
    },
]

metric_rows = []
for scenario in scenarios:
    delete_rows_for_query(scenario["query"])
    metric_rows.append(run_and_measure(scenario["mode"], scenario["client"], scenario["query"], "1"))
    metric_rows.append(run_and_measure(
        scenario["mode"],
        scenario["client"],
        scenario["query"],
        "2",
        sleep_before=scenario["sleep_second_run"],
    ))

display_table(
    metric_rows,
    ["mode", "run", "latency_ms", "local", "foreign", "rows_delta", "rows_total", "interpretation"],
    "Compact retrieval-mode metrics",
)

Deleted 0 old demo rows for this query.
Deleted 0 old demo rows for this query.
Deleted 0 old demo rows for this query.


### Compact retrieval-mode metrics

| mode | run | latency_ms | local | foreign | rows_delta | rows_total | interpretation |
| --- | --- | --- | --- | --- | --- | --- | --- |
| hybrid_search | 1 | 2514.0 | 0 | 3 | 3 | 3 | Tavily supplied fresh results |
| hybrid_search | 2 | 710.5 | 0 | 3 | 3 | 6 | Tavily supplied fresh results |
| freshness_cache | 1 | 7.7 | 3 | 0 | 0 | 0 | Oracle reused cache/memory |
| freshness_cache | 2 | 5.9 | 3 | 0 | 0 | 0 | Oracle reused cache/memory |
| cache_then_memory | 1 | 9.0 | 3 | 0 | 0 | 0 | Oracle reused cache/memory |
| cache_then_memory | 2 | 21.8 | 3 | 0 | 0 | 0 | Oracle reused cache/memory |

## Inspect persisted rows by scenario

This makes it easy to verify which mode wrote what into Oracle.

In [3]:
for scenario in scenarios:
    show_persisted_rows(scenario["query"], f"Persisted rows: {scenario['mode']}")

### Persisted rows: hybrid_search

| memory_scope | retrieval_mode | cache_hit | query_count | source_title | preview |
| --- | --- | --- | --- | --- | --- |
| cache_plus_memory | hybrid_search | 0 | 1 | Understand Hybrid Vector Indexes -... | A hybrid vector index inherits all the information retrieval capabilities of Oracle... |
| cache_plus_memory | hybrid_search | 0 | 1 | Hybrid Vector Search Index – An... | ### Hybrid Vector Search Index – An example. These allow us to perform similarity... |
| cache_plus_memory | hybrid_search | 0 | 1 | Understand Hybrid Search | With hybrid search, you can search through your documents by performing a combination... |
| cache_plus_memory | hybrid_search | 0 | 1 | Understand Hybrid Search | With hybrid search, you can search through your documents by performing a combination... |
| cache_plus_memory | hybrid_search | 0 | 1 | Understand Hybrid Vector Indexes -... | A hybrid vector index inherits all the information retrieval capabilities of Oracle... |
| cache_plus_memory | hybrid_search | 0 | 1 | Hybrid Vector Search Index – An... | ### Hybrid Vector Search Index – An example. These allow us to perform similarity... |

### Persisted rows: freshness_cache

No rows.

### Persisted rows: cache_then_memory

No rows.

## Cleanup

Run this at the end. It deletes only the rows created by this benchmark notebook.

In [4]:
for scenario in scenarios:
    delete_rows_for_query(scenario["query"])
print("Cleaned evaluation metric demo rows.")

Deleted 6 old demo rows for this query.
Deleted 0 old demo rows for this query.
Deleted 0 old demo rows for this query.
Cleaned evaluation metric demo rows.
